# 03 — Athena: Serverless Query Pipeline

Amazon Athena lets you query S3 data using standard SQL. You pay per TB scanned.

**Key moto limitation:** Athena in moto does NOT execute SQL. To test result parsing, you inject fake results via a REST endpoint. This notebook teaches both the real AWS API *and* the moto injection pattern.

- Section A: Workgroups
- Section B: Named queries
- Section C: Query execution lifecycle (the core interview topic)
- Section D: Parsing results into a DataFrame
- Section E: Moto result injection
- Section F: Data catalogs

In [1]:
import sys, time
sys.path.insert(0, "..")
import helpers

import boto3
import pandas as pd
from moto import mock_aws

---
## Section A — Workgroups

A workgroup isolates queries by team/environment and controls where results are stored.

In [2]:
# A1: Create a workgroup with a default output location
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="athena-results")

    athena.create_work_group(
        Name="de-team",
        Configuration={
            "ResultConfiguration": {
                "OutputLocation": "s3://athena-results/de-team/"
            },
            "EnforceWorkGroupConfiguration": True,
        },
        Description="Data Engineering team workgroup",
    )

    wg = athena.get_work_group(WorkGroup="de-team")["WorkGroup"]
    print("Workgroup:", wg["Name"])
    print("Output:", wg["Configuration"]["ResultConfiguration"]["OutputLocation"])

    all_wgs = [w["Name"] for w in athena.list_work_groups()["WorkGroups"]]
    print("All workgroups:", all_wgs)

Workgroup: de-team
Output: s3://athena-results/de-team/
All workgroups: ['primary', 'de-team']


---
## Section B — Named Queries

In [3]:
# B1: Create and retrieve named queries (reusable SQL templates)
# Note: "primary" workgroup already exists in moto by default — use a different name.
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    athena.create_work_group(Name="named-queries-wg", Configuration={})

    resp = athena.create_named_query(
        Name="daily-order-count",
        Description="Count orders per day",
        Database="analytics_db",
        QueryString="SELECT dt, COUNT(*) AS order_count FROM orders GROUP BY dt ORDER BY dt",
        WorkGroup="named-queries-wg",
    )
    query_id = resp["NamedQueryId"]

    nq = athena.get_named_query(NamedQueryId=query_id)["NamedQuery"]
    print("Named query:", nq["Name"])
    print("SQL:", nq["QueryString"][:60], "...")

Named query: daily-order-count
SQL: SELECT dt, COUNT(*) AS order_count FROM orders GROUP BY dt O ...


---
## Section C — Query Execution Lifecycle

This is the **most important pattern** for Athena interviews:
1. `start_query_execution` → get an execution ID
2. Poll `get_query_execution` until `Status.State` is `SUCCEEDED` or `FAILED`
3. `get_query_results` → parse the results

In [4]:
# C1: Start a query and get the execution ID
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="query-results")

    resp = athena.start_query_execution(
        QueryString="SELECT * FROM orders LIMIT 10",
        QueryExecutionContext={"Database": "analytics_db"},
        ResultConfiguration={"OutputLocation": "s3://query-results/"},
    )
    exec_id = resp["QueryExecutionId"]
    print("Execution ID:", exec_id)

Execution ID: a674fd8e-cd8d-435b-bf39-79fb91ced644


In [5]:
# C2: Naive polling loop — wait for SUCCEEDED or FAILED
def wait_for_query(athena_client, execution_id: str, max_attempts: int = 20) -> str:
    """Poll until the query finishes. Returns the final state string."""
    terminal_states = {"SUCCEEDED", "FAILED", "CANCELLED"}
    for attempt in range(max_attempts):
        status = athena_client.get_query_execution(QueryExecutionId=execution_id)
        state = status["QueryExecution"]["Status"]["State"]
        print(f"  Attempt {attempt+1}: {state}")
        if state in terminal_states:
            if state == "FAILED":
                reason = status["QueryExecution"]["Status"].get("StateChangeReason", "unknown")
                print(f"  FAILED reason: {reason}")
            return state
        time.sleep(0.5)  # exponential backoff in production
    raise TimeoutError(f"Query {execution_id} did not finish in {max_attempts} attempts")

with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="results")

    exec_id = athena.start_query_execution(
        QueryString="SELECT COUNT(*) FROM events",
        QueryExecutionContext={"Database": "demo_db"},
        ResultConfiguration={"OutputLocation": "s3://results/"},
    )["QueryExecutionId"]

    final_state = wait_for_query(athena, exec_id)
    print("Final state:", final_state)

  Attempt 1: SUCCEEDED
Final state: SUCCEEDED


In [6]:
# C3: Stop a running query
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="results2")

    exec_id = athena.start_query_execution(
        QueryString="SELECT * FROM large_table",
        QueryExecutionContext={"Database": "demo_db"},
        ResultConfiguration={"OutputLocation": "s3://results2/"},
    )["QueryExecutionId"]

    athena.stop_query_execution(QueryExecutionId=exec_id)
    state = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"]["State"]
    print("State after stop:", state)  # CANCELLED

State after stop: CANCELLED


---
## Section D — Parsing Query Results

**Critical:** `get_query_results` returns the header row as the FIRST row in `ResultSet.Rows`. You must skip it.

In [7]:
# D1: What the raw response looks like (moto vs real Athena)
#
# Real Athena:  first row in Rows IS the column headers (duplicated from ColumnInfo)
# moto:         no header row in Rows — column names come only from ResultSetMetadata.ColumnInfo
#
# The helpers.parse_athena_results() handles both cases automatically.

EXAMPLE_MOTO_RESPONSE = {
    "ResultSet": {
        "Rows": [
            # moto: only data rows, no header row
            {"Data": [{"VarCharValue": "Alice"}, {"VarCharValue": "30"}]},
            {"Data": [{"VarCharValue": "Bob"},   {"VarCharValue": "25"}]},
        ],
        "ResultSetMetadata": {
            "ColumnInfo": [
                {"Name": "name", "Type": "varchar"},
                {"Name": "age",  "Type": "integer"},
            ]
        },
    }
}

EXAMPLE_REAL_ATHENA_RESPONSE = {
    "ResultSet": {
        "Rows": [
            # Real Athena: first row = column names
            {"Data": [{"VarCharValue": "name"}, {"VarCharValue": "age"}]},
            {"Data": [{"VarCharValue": "Alice"}, {"VarCharValue": "30"}]},
            {"Data": [{"VarCharValue": "Bob"},   {"VarCharValue": "25"}]},
        ],
        "ResultSetMetadata": {
            "ColumnInfo": [
                {"Name": "name", "Type": "varchar"},
                {"Name": "age",  "Type": "integer"},
            ]
        },
    }
}

print("moto response → parse_athena_results:")
print(helpers.parse_athena_results(EXAMPLE_MOTO_RESPONSE))

print("\nReal Athena response → parse_athena_results:")
print(helpers.parse_athena_results(EXAMPLE_REAL_ATHENA_RESPONSE))

print("\nBoth produce the same DataFrame — the helper handles both automatically.")

moto response → parse_athena_results:


    name age
0  Alice  30
1    Bob  25

Real Athena response → parse_athena_results:
    name age
0  Alice  30
1    Bob  25

Both produce the same DataFrame — the helper handles both automatically.


---
## Section E — moto Result Injection

Because moto's Athena doesn't run SQL, you inject the results you want before running the query. The next `get_query_results` call returns those injected rows.

In [8]:
# E1: Full injection + query + parse pattern
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="results3")

    # Step 1: Inject the results you WANT the query to return
    helpers.inject_athena_results(
        rows=[
            ["Bangkok", "1500"],
            ["Singapore", "2200"],
            ["Tokyo", "1800"],
        ],
        column_info=[
            {"Name": "city",       "Type": "varchar"},
            {"Name": "order_count","Type": "integer"},
        ],
    )

    # Step 2: Run the query
    exec_id = athena.start_query_execution(
        QueryString="SELECT city, COUNT(*) AS order_count FROM orders GROUP BY city",
        QueryExecutionContext={"Database": "analytics_db"},
        ResultConfiguration={"OutputLocation": "s3://results3/"},
    )["QueryExecutionId"]

    # Step 3: Wait (moto finishes immediately)
    wait_for_query(athena, exec_id)

    # Step 4: Get and parse results
    raw_results = athena.get_query_results(QueryExecutionId=exec_id)
    df = helpers.parse_athena_results(raw_results)
    print(df)
    assert list(df.columns) == ["city", "order_count"]
    assert len(df) == 3
    print("\nAssertion passed: 3 rows, correct columns")

  Attempt 1: SUCCEEDED
        city order_count
0    Bangkok        1500
1  Singapore        2200
2      Tokyo        1800

Assertion passed: 3 rows, correct columns


In [9]:
# E2: List all query executions
with mock_aws():
    athena = helpers.make_boto3_client("athena")
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="multi-results")

    for sql in ["SELECT 1", "SELECT 2", "SELECT 3"]:
        athena.start_query_execution(
            QueryString=sql,
            QueryExecutionContext={"Database": "db"},
            ResultConfiguration={"OutputLocation": "s3://multi-results/"},
        )

    exec_ids = athena.list_query_executions()["QueryExecutionIds"]
    print(f"Total executions: {len(exec_ids)}")

Total executions: 3


---
## Section F — Data Catalogs

In [10]:
# F1: Register an Athena data catalog (connects to a Glue catalog)
with mock_aws():
    athena = helpers.make_boto3_client("athena")

    athena.create_data_catalog(
        Name="my-glue-catalog",
        Type="GLUE",
        Description="Glue Data Catalog for analytics",
        Parameters={"catalog-id": "123456789012"},
    )

    catalogs = athena.list_data_catalogs()["DataCatalogsSummary"]
    print("Catalogs:", [(c["CatalogName"], c["Type"]) for c in catalogs])

Catalogs: [('my-glue-catalog', 'GLUE')]


## Summary

| API Call | Purpose |
|---|---|
| `create_work_group` | Isolate queries; set default output S3 location |
| `create_named_query` | Save a reusable SQL query |
| `start_query_execution` | Start a query; returns an execution ID |
| `get_query_execution` | Poll status — check `Status.State` |
| `get_query_results` | Fetch result rows — **skip first row (header)** |
| `stop_query_execution` | Cancel a running query |
| `list_query_executions` | List all execution IDs |
| `create_data_catalog` | Register an external catalog (e.g. Glue) |

**Polling best practices:**
- Always guard with `max_attempts` to prevent infinite loops
- Use exponential backoff in production (`time.sleep(2**attempt)`)
- Check `FAILED` and log `StateChangeReason`

**Next notebook:** [04_ec2_fleet_management.ipynb](04_ec2_fleet_management.ipynb)